In [14]:
#Data source: [BIXI Montréal Open Data Portal](https://bixi.com/en/open-data/)

In [19]:
import padas as pd

print("Summary 2025 Bixi dataset...")
df = pd.read_csv('../data/raw/bixi_2025_raw.csv')

print(f"Total Rows: {len(df):,}")
print(f"Total Columns: {len(df.columns)}")

print("\n SCHEMA & NULL VALUE")
print(df.info(show_counts=True))

print("\n MISSING VALUES PER COLUMN")
print(df.isnull().sum())

df.head()

Summary 2025 Bixi dataset...
Total Rows: 14,249,363
Total Columns: 10

--- SCHEMA & NULL VALUE ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 14249363 entries, 0 to 14249362
Data columns (total 10 columns):
 #   Column                      Non-Null Count     Dtype  
---  ------                      --------------     -----  
 0   STARTSTATIONNAME            14249363 non-null  object 
 1   STARTSTATIONARRONDISSEMENT  14249341 non-null  object 
 2   STARTSTATIONLATITUDE        14249363 non-null  float64
 3   STARTSTATIONLONGITUDE       14249363 non-null  float64
 4   ENDSTATIONNAME              14197540 non-null  object 
 5   ENDSTATIONARRONDISSEMENT    14196502 non-null  object 
 6   ENDSTATIONLATITUDE          14197540 non-null  float64
 7   ENDSTATIONLONGITUDE         14197540 non-null  float64
 8   STARTTIMEMS                 14249363 non-null  int64  
 9   ENDTIMEMS                   14197541 non-null  float64
dtypes: float64(5), int64(1), object(4)
memory usage: 1.1+ GB
None

,STARTSTATIONNAME,STARTSTATIONARRONDISSEMENT,STARTSTATIONLATITUDE,STARTSTATIONLONGITUDE,ENDSTATIONNAME,ENDSTATIONARRONDISSEMENT,ENDSTATIONLATITUDE,ENDSTATIONLONGITUDE,STARTTIMEMS,ENDTIMEMS
0,Parc Émilie-Gamelin (St-Hubert / de Maisonneuv...,Ville-Marie,45.515868,-73.560084,NaN,NaN,NaN,NaN,1741120865258,NaN
1,Métro Mont-Royal (Utilités publiques / Rivard),Le Plateau-Mont-Royal,45.524247,-73.581662,NaN,NaN,NaN,NaN,1741144108111,NaN
2,Notre-Dame / St-Martin,Le Sud-Ouest,45.488302,-73.568718,NaN,NaN,NaN,NaN,1741094909506,NaN
3,de Maisonneuve / Greene,Westmount,45.486971,-73.589293,NaN,NaN,NaN,NaN,1741098869605,NaN
4,Métro Sherbrooke (de Rigaud / Berri),Le Plateau-Mont-Royal,45.518143,-73.568004,NaN,NaN,NaN,NaN,1741094527644,NaN


In [16]:
df = pd.read_csv('../data/raw/bixi_2025_raw.csv')

df['duration_sec'] = (df['ENDTIMEMS'] - df['STARTTIMEMS']) / 1000

df['start_date'] = pd.to_datetime(df['STARTTIMEMS'], unit='ms')
df['end_date'] = pd.to_datetime(df['ENDTIMEMS'], unit='ms')

df['start_hour'] = df['start_date'].dt.hour
df['day_of_week'] = df['start_date'].dt.day_name()
df['month'] = df['start_date'].dt.month

# Remove false starts (< 60s) and unreturned rides (> 4 hours)
df_clean = df[(df['duration_sec'] >= 60) & (df['duration_sec'] <= 14400)].copy()

df_clean['is_round_trip'] = (df_clean['STARTSTATIONNAME'] == df_clean['ENDSTATIONNAME'])

print(f"Original Row Count: {len(df):,}")
print(f"Cleaned Row Count:  {len(df_clean):,}")
print(f"Removed Anomalies:  {len(df) - len(df_clean):,} rows")

Original Row Count: 14,249,363
Cleaned Row Count:  13,990,411
Removed Anomalies:  258,952 rows


In [21]:
import os

os.makedirs('../data/processed', exist_ok=True)

output_path = '../data/processed/bixi_2025_cleaned.csv'
print(f"Exporting cleaned dataset to {output_path}...")

df_clean.to_csv(output_path, index=False)

file_size_mb = os.path.getsize(output_path) / (1024 * 1024)
print(f"Export Complete! File Size: {file_size_mb:.2f} MB")

Exporting cleaned dataset to ../data/processed/bixi_2025_cleaned.csv...
Export Complete! File Size: 3650.98 MB
